## 10. Resolution ablation — 640 / 768 / 896 / 1024 px (Phase 7, §24B)

Add these cells **at the end** of `vigraph_qlora_windows.ipynb` (after section 9). Before running
them, run **section 1** (settings + helpers). Training is not repeated, and the dataset must
already exist (section 2).

**Why:** on the hardest diagrams (L4) the fine-tuned model often *runs away*: it keeps listing
invented nodes, or far too many edges, until the token limit. Section 9 showed that more tokens
don't help (4096 gave the same result as 2048). One suspect is image resolution: L4 diagrams are
large (longest side ~1,760 px on average) and are shrunk the most before the model sees them.

These runs evaluate the **same fine-tuned adapter** on the **same 112 test samples** with greedy
decoding and **2048 tokens**, and change only `image_max_side`, the longest side an image is
shrunk to (smaller images are never enlarged).

- Only the evaluation size changes. The adapter was trained at 896 px, so 640 and 768 give the
  model fewer image tokens than in training, and 1024 more.
- **896 px** is the size section 7 already evaluated with exactly these settings: cell 10.4 checks
  that and reuses that run (no GPU time). If the settings differ, it runs a new `-px896` run.
- Run the sections **one at a time, in order**. Each run resumes after a stop: run section 1 and
  10.1, then the same cell again.
- Expect roughly **2–3 h per size** (the first 896 px run took 2.5 h; smaller images are faster,
  1024 is slower), about 7–8 h for the three new sizes.
- New run names end in `-px640-s0`, `-px768-s0` and `-px1024-s0`; earlier results stay untouched.

In [ ]:
# 10.1 Settings for the resolution ablation (uses section 1's helpers)
import gzip
import yaml

SIZES = [640, 768, 896, 1024]           # image_max_side values, run in this order
TRAINED_SIZE = image_side()             # 896: the memory probe's choice, used for training
MAX_NEW_TOKENS_RES = 2048               # same budget as section 7 (section 9: 4096 changed nothing)
EXPECTED = EVAL_LIMIT or 500
model_cfg = yaml.safe_load(CONFIG.read_text(encoding="utf-8"))["model"]
hf_res = ["--backend", "hf", "--model-id", model_cfg["id"], "--dtype", COMPUTE_DTYPE,
          "--device-map", "cuda:0", "--max-new-tokens", MAX_NEW_TOKENS_RES,
          "--adapter", RUN_DIR / "adapter"]
if model_cfg.get("revision"):
    hf_res += ["--revision", model_cfg["revision"]]
FIRST_RUN = EVAL_DIR / f"{RUN_NAME}-s0"          # section 7's fine-tuned run
COMPARE_DIR = EVAL_DIR / "comparisons-resolution"


def finished(path):
    """True when a run has scored all EXPECTED samples."""
    if not ((path / "summary.json").exists() and (path / "run.json").exists()):
        return False
    info = read_json(path / "run.json")
    return bool(info.get("completed_at")) and info.get("samples_done") == EXPECTED


def reusable(path, size):
    """True when a finished run is the fine-tuned model at this size with these exact settings."""
    if not finished(path):
        return False
    config = read_json(path / "run.json")["config"]
    return (bool(config.get("adapter")) and config.get("image_max_side") == size
            and config.get("max_new_tokens") == MAX_NEW_TOKENS_RES
            and config.get("temperature") == 0.0 and config.get("limit") == EVAL_LIMIT)


def run_path(size):
    return FIRST_RUN if reusable(FIRST_RUN, size) else EVAL_DIR / f"{RUN_NAME}-px{size}-s0"


def truncation_by_level(path):
    """{level: (samples, cut off at the token limit, failed)} from a run's predictions."""
    counts = {}
    raw = path / "predictions.jsonl"
    text = raw.read_text(encoding="utf-8") if raw.exists() else gzip.decompress(
        (path / "predictions.jsonl.gz").read_bytes()).decode("utf-8")
    for line in text.splitlines():
        if not line.strip():
            continue
        result = json.loads(line)
        level = result["attributes"]["level"]
        analysis = result["prediction"].get("analysis") or {}
        attempts = analysis.get("extraction", {}).get("attempts", [])
        cut = any(a["output"].get("finish_reason") == "length" for a in attempts)
        failed = not result["scores"]["has_graph"]
        n, t, f = counts.get(level, (0, 0, 0))
        counts[level] = (n + 1, t + cut, f + failed)
    return counts


def quick_summary(size):
    path = run_path(size)
    macro = read_json(path / "summary.json")["macro"]
    n, cut, failed = truncation_by_level(path).get(4, (0, 0, 0))
    print(f"{size} px: node F1 {macro['node_f1']['mean']:.3f} | edge F1 {macro['edge_f1']['mean']:.3f}"
          f" | graph sim {macro['graph_similarity']['mean']:.3f} | L4 cut off {cut}/{n}, failed {failed}")


def evaluate_size(size):
    """Run (or resume) the fine-tuned model at one image size; skip it when finished."""
    out = run_path(size)
    if finished(out):
        print(f"{size} px: done → {out.name}")
        quick_summary(size)
        return out
    command = ["module", "evaluation", "run", "--dataset", DATA_DIR, "--split", "test",
               "--out", out, "--image-max-side", size, *hf_res]
    if EVAL_LIMIT:
        command += ["--limit", EVAL_LIMIT]
    started = time.time()
    driver(*command, log=f"eval-{out.name}")
    errors = read_json(out / "run.json").get("errors", 0)
    print(f"\n{size} px finished ({(time.time() - started) / 3600:.2f} h in this session)")
    if errors:
        print(f"Note: {errors} sample(s) raised an error; they count as failures.")
    quick_summary(size)
    return out


if not (RUN_DIR / "adapter" / "adapter_config.json").exists():
    raise RuntimeError(f"No trained adapter in {RUN_DIR / 'adapter'} — section 6 must have finished.")
if TRAINED_SIZE not in SIZES:
    raise RuntimeError(f"The adapter was trained at {TRAINED_SIZE} px; add it to SIZES.")
print(f"max_new_tokens {MAX_NEW_TOKENS_RES} | {EXPECTED} test samples | trained at {TRAINED_SIZE} px")
for size in SIZES:
    path = run_path(size)
    state = ("done" if finished(path) else
             "started — the next run resumes it" if (path / "predictions.jsonl").exists() else "to run")
    print(f"  {size:>4} px → {path.name}: {state}")

### 10.2 640 px

Smallest size: about half the image tokens of 896 px. The quickest of the four runs.

In [ ]:
# 10.2 Fine-tuned model at 640 px (resumable)
run_640 = evaluate_size(640)

### 10.3 768 px

Between 640 and the training size.

In [ ]:
# 10.3 Fine-tuned model at 768 px (resumable)
run_768 = evaluate_size(768)

### 10.4 896 px

The training size. Section 7's fine-tuned run already used exactly these settings, so this cell normally just prints **done** and its scores. If it starts a new run, the settings of section 7's run differ (e.g. a different `EVAL_LIMIT`); let it run.

In [ ]:
# 10.4 Fine-tuned model at 896 px (resumable)
run_896 = evaluate_size(896)

### 10.5 1024 px

Largest size: more image tokens than in training, and the slowest run. It fits the VRAM limit easily (evaluation needs far less memory than training). If a sample reports `CUDA out of memory`, send the output.

In [ ]:
# 10.5 Fine-tuned model at 1024 px (resumable)
run_1024 = evaluate_size(1024)

### 10.6 – 10.8 Compare, summarize, hand back

Run these after the four sizes (they skip whatever isn't finished).

In [ ]:
# 10.6 Paired comparisons (§20.11): each size vs the training size, on the same 112 samples
COMPARE_DIR.mkdir(parents=True, exist_ok=True)
reference = run_path(TRAINED_SIZE)
for size in SIZES:
    if size == TRAINED_SIZE:
        continue
    other = run_path(size)
    missing = [f"{s} px" for s, p in ((TRAINED_SIZE, reference), (size, other)) if not finished(p)]
    if missing:
        print(f"skipped {size} px vs {TRAINED_SIZE} px: not finished yet: {', '.join(missing)}")
        continue
    driver("module", "evaluation", "compare", "--a", reference, "--b", other,
           "--out", COMPARE_DIR / f"px{size}-vs-px{TRAINED_SIZE}.md", log="compare-resolution")

In [ ]:
# 10.7 Results by image size (reads the run files only) — paste this output back with the zip
keys = ("node_f1", "edge_f1", "edge_strict_f1", "graph_similarity", "label_accuracy",
        "qa_accuracy", "valid_first_attempt", "valid_post_repair")
header = ("size", "node F1", "edge F1", "strict", "graph sim", "labels", "QA", "valid@1", "valid",
          "s/sample")
print(" | ".join(header))
for size in SIZES:
    path = run_path(size)
    if not finished(path):
        print(f"{size} px | not finished")
        continue
    summary = read_json(path / "summary.json")
    values = [summary["macro"].get(k, {}).get("mean") for k in keys]
    latency = summary.get("latency_ms", {}).get("mean")
    print(" | ".join([f"{size} px", *("–" if v is None else f"{v:.3f}" for v in values),
                      "–" if latency is None else f"{latency / 1000:.0f}"]))

print("\ncut off at the token limit / failed, per level (of n):")
for size in SIZES:
    path = run_path(size)
    if finished(path):
        counts = truncation_by_level(path)
        print(f"  {size:>4} px  " + "  ".join(
            f"L{lv}: {t}/{f} of {n}" for lv, (n, t, f) in sorted(counts.items())))

In [ ]:
# 10.8 Hand-back: one small zip (no adapter — only the new evaluations and comparisons)
handback_res = OUT / "handback" / f"vigraph-handback-{RUN_NAME}-resolution.zip"
new_runs = [run_path(s) for s in SIZES if run_path(s) != FIRST_RUN and finished(run_path(s))]
extra_res = sorted(COMPARE_DIR.glob("*")) if COMPARE_DIR.exists() else []
driver("handback", "--out", handback_res, "--eval", *new_runs, "--extra", *extra_res, log="handback")
print("\nGive back:", handback_res)

### Files to give back

**`F:\vigraph\output\handback\vigraph-handback-qwen3vl-2b-qlora-a6000-v1-resolution.zip`**: a
few hundred KB, small enough to upload directly. It contains the new evaluation runs (640, 768
and 1024 px; the 896 px run is already in the repo), each with `run.json`, `summary.json`,
`report.md` and `predictions.jsonl.gz`, the three comparisons against 896 px, and
`MANIFEST.json`. Also paste the output of cell 10.7.

If a run stopped with an error, send the output of its cell, or the matching
`F:\vigraph\output\logs\eval-…-px<size>-s0.log`.